# Explore CHAOS CT/MRI

This notebook uses the standard PyHealth flow: CHAOS dataset → `set_task()` → `SampleDataset` → `get_dataloader()`. The image is a DICOM series and the mask is a directory of PNG slices. Both are lazy: `set_task()` only records paths and metadata, and `show_chaos_samples()` can overlay the mask on the image when a sample is read. Data is read from `PYHEALTH_IMAGING_DATA` (default `~/pyhealth-imaging-data`); `download=True` fetches CHAOS automatically.

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from pyhealth.datasets import CHAOSDataset, get_dataloader

DATA_ROOT = Path(
    os.environ.get("PYHEALTH_IMAGING_DATA", "~/pyhealth-imaging-data")
).expanduser()
# CHAOS is downloaded into DATA_ROOT when download=True.
CACHE_DIR = Path(
    os.environ.get("PYHEALTH_IMAGING_CACHE", "~/.cache/pyhealth-imaging")
).expanduser()
# Processed caches and generated metadata are stored here, outside the repo.
METADATA_DIR = Path(CACHE_DIR) / "metadata"

dataset = CHAOSDataset(
    root=DATA_ROOT / "CHAOS",
    cache_dir=CACHE_DIR,
    metadata_dir=str(METADATA_DIR / "chaos"),
    download=True,
)
dataset.stats()


def show_chaos_samples(samples, indices, overlay_mask=False):
    rows = 1 if overlay_mask else 2
    fig, axes = plt.subplots(
        rows, len(indices), figsize=(4 * len(indices), 4 * rows), squeeze=False
    )
    for col, index in enumerate(indices):
        sample = samples[index]
        slice_idx = sample["image"].shape[1] // 2
        image = sample["image"][0, slice_idx].cpu()
        mask = sample["mask"][0, slice_idx].cpu()
        if overlay_mask:
            axes[0, col].imshow(image, cmap="gray")
            axes[0, col].imshow(
                np.ma.masked_where(mask == 0, mask), cmap="autumn", alpha=0.35
            )
            axes[0, col].set_title(sample["record_id"])
        else:
            axes[0, col].imshow(image, cmap="gray")
            axes[0, col].set_title(sample["record_id"])
            axes[1, col].imshow(mask, cmap="gray")
            axes[1, col].set_title("mask")
        for row in range(rows):
            axes[row, col].axis("off")
    plt.show()


In [ ]:
patient_ids = dataset.unique_patient_ids
ct_id = next(pid for pid in patient_ids if pid.startswith("ct_"))
mr_id = next(pid for pid in patient_ids if pid.startswith("mr_"))
print("CT:", ct_id)
print("MR:", mr_id)
dataset.get_patient(ct_id).get_events()[0].attr_dict


In [ ]:
task = dataset.default_task
print(task.task_name)
print(task.input_schema)
print(task.output_schema)


In [ ]:
samples = dataset.set_task(task, num_workers=1)
print("samples:", len(samples))
print("input schema:", samples.input_schema)
print("output schema:", samples.output_schema)


In [ ]:
sample = samples[0]
print(sample.keys())
print("image:", sample["image"].shape, sample["image"].dtype)
print("mask:", sample["mask"].shape, sample["mask"].dtype)
print("mask labels:", sorted(sample["mask"].unique().tolist()))


In [ ]:
loader = get_dataloader(samples, batch_size=1)
batch = next(iter(loader))
print("batch image:", batch["image"].shape)
print("batch mask:", batch["mask"].shape)


In [ ]:
record_ids = [samples[i]["record_id"] for i in range(len(samples))]

def find_index(token):
    return next(i for i, record_id in enumerate(record_ids) if token in record_id)

indices = []
for token in ["ct_", "_t1dual_inphase", "_t2spir"]:
    try:
        indices.append(find_index(token))
    except StopIteration:
        pass

show_chaos_samples(samples, indices, overlay_mask=True)
